# SPK-8 — Caching & Persistence Tradeoffs

**Break → Detect → Fix → Prove.** Caching is **not free** — `.cache()` trades **memory** for
avoided **recompute**, and it only pays off when a frame is **reused multiple times**, at a
storage level that fits. Cache a single-use frame, or a `MEMORY_ONLY` frame too big for memory,
or forget to `unpersist()`, and the cache does nothing, thrashes (evict → recompute), or pins
memory more useful data needed.

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and keep the **Storage** tab in view —
that is where caching becomes visible.

**Laptop-safe:** the data is *generated lazily* and only `count()`-ed — never collected or
written — so nothing fills disk. We never try to crash the box; we make the *cost* of caching
visible. Runs fine on the default **tuned** box (no need for `make up-constrained`). Every cached
frame is `unpersist()`-ed at the end.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md) (the **Storage** tab), and the
[troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import uniform_keys, wide_rows
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F
from pyspark import StorageLevel

# Caching shows up in the Storage tab — keep it open while these cells run.
print("Spark UI:", "http://localhost:4040", "->  Storage tab")
spark

## Step 0 — Parameters & an expensive derived frame

We build one **expensive** intermediate frame — a wide projection over a large generated fact,
plus a per-key aggregation — that we will reuse several times. It is generated *lazily*, so
defining it is free; the cost is paid only on an action (`.count()`). This is the frame whose
recompute-vs-cache tradeoff the whole module is about.

We run on the **tuned** profile throughout — this module is about DataFrame caching, not the
session safety nets.

In [ ]:
# Lower N_ROWS to ~10M if your laptop is slow; raise it to make recompute cost more dramatic.
N_ROWS  = 15_000_000
N_KEYS  = 2_000
N_REUSE = 4          # how many times we 'reuse' the derived frame in the speedup demo

apply_profile(spark, "tuned")

# An expensive lineage: wide rows -> projection/arithmetic -> per-key aggregation.
base    = wide_rows(spark, n_rows=N_ROWS, n_cols=30)
heavy   = (base
           .withColumn("score", sum(F.col(f"c{i}") for i in range(30)))
           .groupBy((F.pmod(F.col("row_id"), F.lit(N_KEYS))).alias("key"))
           .agg(F.sum("score").alias("score_sum"),
                F.avg("score").alias("score_avg"),
                F.count(F.lit(1)).alias("n")))

print(f"heavy: per-key aggregate over ~{N_ROWS:,} wide rows (lazy — nothing computed yet)")

## 1. Break it (A) — `.cache()` is lazy: first access still computes

`df.cache()` only *marks* the frame to be kept — it does **nothing** until an action. Right after
`.cache()` the **Storage** tab is **empty**. The **first** action computes the full lineage *and*
populates the cache (so it's no faster than uncached). **Subsequent** actions read the cache and
are fast.

Watch the Storage tab: it stays empty until the first `count()` below, then `heavy` appears with
**Fraction Cached = 100%**.

In [ ]:
heavy.cache()                      # lazy — Storage tab is still EMPTY at this point
print("right after .cache():  nothing in the Storage tab yet (cache is lazy)\n")

m_first  = measure(spark, "cache: 1st access (computes + fills cache)", lambda: heavy.count())
m_second = measure(spark, "cache: 2nd access (cache hit)",             lambda: heavy.count())

print(f"\n1st access runtime : {m_first['runtime_s']} s   <- full lineage, then cache filled")
print(f"2nd access runtime : {m_second['runtime_s']} s   <- cache hit, much faster")
print("Storage tab now shows 'heavy' resident -> check Fraction Cached = 100%.")

## 2. Break it (B) — reuse: recompute every time vs cache once

The derived frame is reused `N_REUSE` times (here, several filtered counts). **Uncached**, Spark
recomputes the whole lineage on every reuse. **Cached**, it computes once and replays. We
`measure()` the **total** wall-clock across all reuses for each, then `compare()` them — the
headline proof.

First the **uncached** baseline (we `unpersist()` so each reuse pays the full recompute), then the
cached run.

In [ ]:
def reuse_all(df):
    # Touch the frame N_REUSE times with cheap, different actions (small results — safe).
    total = 0
    for t in range(N_REUSE):
        total += df.filter(F.col("key") % N_REUSE == t).count()
    return total

heavy.unpersist()                                  # uncached: each reuse recomputes the lineage
m_uncached = measure(spark, "reused x%d (uncached)" % N_REUSE, lambda: reuse_all(heavy))
print("uncached total:", m_uncached['runtime_s'], "s  (lineage recomputed every reuse)")

In [ ]:
heavy.cache()
heavy.count()                                      # materialize once so the reuse loop hits the cache
m_cached = measure(spark, "reused x%d (cached)" % N_REUSE, lambda: reuse_all(heavy))
print("cached total  :", m_cached['runtime_s'], "s  (lineage computed once, then replayed)\n")

print("Prove it — reuse cost, uncached vs cached (the win is bought with the memory the cache holds):")
compare([m_uncached, m_cached])
heavy.unpersist()                                  # release before the storage-level demo

## 3. Break it (C) — storage levels: `MEMORY_ONLY` vs `MEMORY_AND_DISK` vs `DISK_ONLY`

`persist(StorageLevel.X)` chooses *where* cached partitions live:

- **`MEMORY_ONLY`** — deserialized in memory. If the frame doesn't fully fit, the surplus is
  **evicted and recomputed on every access** → GC / recompute **thrash** (Fraction Cached < 100%
  in the Storage tab). Can be *slower* than not caching.
- **`MEMORY_AND_DISK`** — overflow spills to **disk** instead of being recomputed (a disk read,
  not a lineage replay). The safe default for big, reused frames.
- **`DISK_ONLY`** — always on disk: cheap on memory, slower than RAM, but never recomputes.

After each run, check the **Storage** tab: **Storage Level**, **Fraction Cached**, and
**Size in Memory vs Size on Disk**. (`persist()` needs an `unpersist()` before re-persisting the
same frame at a different level — the helper does that between runs.)

In [ ]:
def time_level(level_name, level):
    heavy.unpersist()                      # clear any prior level
    heavy.persist(level)
    first  = measure(spark, f"{level_name}: 1st access", lambda: heavy.count())
    second = measure(spark, f"{level_name}: 2nd access", lambda: heavy.count())
    print(f"  {level_name:<16} 1st={first['runtime_s']}s  2nd={second['runtime_s']}s")
    return second                          # repeated-access runtime is the comparison point

print("Persisting 'heavy' at three storage levels (watch the Storage tab after each):\n")
m_mem      = time_level("MEMORY_ONLY",     StorageLevel.MEMORY_ONLY)
m_mem_disk = time_level("MEMORY_AND_DISK", StorageLevel.MEMORY_AND_DISK)
m_disk     = time_level("DISK_ONLY",       StorageLevel.DISK_ONLY)

print("\nProve it — repeated-access runtime by storage level:")
compare([m_mem, m_mem_disk, m_disk])
heavy.unpersist()

## 4. Break it (D) — forgetting `unpersist()` pins memory & evicts useful caches

A cached frame keeps holding memory whether or not you still need it. Cache a **single-use**
frame (pure overhead — nothing to amortize against) and *also* leave the useful `heavy` cache
pinned, and the two compete for the same storage memory: the less-useful cache can evict the one
that mattered, forcing it to recompute on its next access.

`unpersist()` releases that memory immediately. Below: cache a throwaway frame, watch **two**
entries pile up in the Storage tab (and Storage Memory climb on the **Executors** tab), then
`unpersist()` the throwaway and see it drop.

In [ ]:
# A frame we will touch exactly once -> caching it is pure overhead.
one_shot = (uniform_keys(spark, n_rows=N_ROWS, n_keys=N_KEYS)
            .groupBy("key").agg(F.sum("amount").alias("amt")))

heavy.cache(); heavy.count()               # the cache that actually matters (reused)
one_shot.cache(); one_shot.count()         # the wasteful cache (used once, never reused)
print("Storage tab now lists TWO frames competing for memory:")
print("  - 'heavy'    : useful (reused)         <- the one we want resident")
print("  - 'one_shot' : wasteful (single-use)   <- pinning memory for nothing")
print("Check the Executors tab: Storage Memory has climbed toward its limit.")

In [ ]:
one_shot.unpersist()                       # release the memory the wasteful cache was pinning
print("After unpersist('one_shot'): it disappears from the Storage tab;")
print("storage memory is freed so 'heavy' (and execution) is no longer crowded out.\n")
print("Headline recap:")
print(f"  lazy cache : 1st {m_first['runtime_s']}s vs 2nd {m_second['runtime_s']}s  (cache hit)")
print(f"  reuse x{N_REUSE}  : uncached {m_uncached['runtime_s']}s vs cached {m_cached['runtime_s']}s")
print("Both wins were BOUGHT with memory -> cache only what you reuse, at a level that fits, then release it.")

## 5. Takeaways & "in real production…"

- **Cache is a tradeoff, not a speedup button** — it spends **memory** to avoid recompute, and
  pays off only when a frame is **reused multiple times**.
- **`.cache()` is lazy** — nothing is cached until the first action; an empty **Storage** tab
  after `.cache()` is the classic "why is it still slow?" gotcha.
- **Pick the storage level by size & reuse:** `MEMORY_ONLY` only if it fits (else it thrashes —
  Fraction Cached < 100%); `MEMORY_AND_DISK` for big reused frames; `DISK_ONLY` when memory's tight.
- **Always `unpersist()`** when the reuse window closes — a forgotten cache evicts caches that
  still matter and feeds executor memory pressure (`SPK-2`).
- **In prod:** cache only well-reused intermediates, default to `MEMORY_AND_DISK` for large ones,
  scope caches to the job that uses them, release them, and watch **Storage Memory** on the
  **Executors** tab so caching doesn't starve execution.

## Teardown

Release every cached frame and clear the cache, then restore the production-tuned safety nets.
Nothing was written (we only counted generated data), so there is nothing else to delete.

In [ ]:
for df in (heavy, one_shot):
    df.unpersist()
spark.catalog.clearCache()           # drop anything still cached
apply_profile(spark, "tuned")        # restore production-tuned safety nets
print("Done. All caches released, profile reset to 'tuned'. "
      "No tables/files were created; `make clean` clears .tmp if needed.")